# Phase 3 — RAG (Arm B)

Thin Colab runner for the **RAG** arm: the base `Qwen/Qwen3.5-4B` model with retrieved Polars 2.0 docs in the prompt's `{context}` block (query = task instruction only, hybrid dense + BM25 search, top-k=5 within ~2,000 tokens).

This notebook only clones the repo, installs pinned dependencies and calls scripts. All logic lives in the repo.

**Runtime:** GPU, target **L4 (24 GB)**, fallback A100. The docs corpus is gitignored, so it is re-downloaded at the pinned Polars commit (`py-2.0.0`).

In [ ]:
# Confirm the GPU (expect an L4 or A100).
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Clone the repo (idempotent) and enter it.
import os
REPO = 'https://github.com/yusuftanriverdi24/Polars-Rag---Finetune.git'
if not os.path.isdir('polars-rag-vs-ft'):
    !git clone -q $REPO polars-rag-vs-ft
%cd polars-rag-vs-ft
!git pull -q

In [ ]:
# Install pinned dependencies: `colab` = GPU stack, `rag` = sentence-transformers +
# LanceDB (exact pins). torch stays Colab's preinstalled CUDA build.
!pip install -q -e ".[colab,rag]"

In [ ]:
# 1. Download the Polars 2.0.0 docs corpus at the pinned commit -> data/raw/.
!python -m polars_llm.rag.ingest

In [ ]:
# 2. Chunk it (API pages + user-guide sections) -> data/rag/chunks.jsonl.
#    The printed sha256 should match `chunks_sha256` in results/rag/chunk_stats.json.
!python -m polars_llm.rag.chunk --stats results/B/chunk_stats_colab.json
!python -c "import json; a=json.load(open('results/rag/chunk_stats.json')); b=json.load(open('results/B/chunk_stats_colab.json')); print('chunks identical to the committed build:', a['chunks_sha256']==b['chunks_sha256'])"

In [ ]:
# 3. Build the LanceDB hybrid index (dense vectors on the GPU + BM25 full-text).
!python -m polars_llm.rag.index --device cuda --batch-size 32

In [ ]:
# 3b. Sanity-check retrieval BEFORE generating: API recall@k of the reference solutions'
#     APIs in the retrieved chunks, overall / per category, plus a k x budget sweep and the
#     longest prompt with context (must fit max_seq_length 4096 with 768 new tokens).
#     If recall at k=5 / 2,000 tokens looks wrong, stop here and adjust --k / --context-budget.
!python -m polars_llm.rag.eval_retrieval --device cuda

In [ ]:
# 4. Generate Arm B for all 120 tasks (retrieval runs first, then the LLM loads).
!python scripts/generate.py --arm B --model Qwen/Qwen3.5-4B --batch-size 8

In [ ]:
# 5. Score on CPU (sandbox execution under Polars 2.0.0).
!python scripts/score.py --arm B

In [ ]:
# 6. Same determinism check every arm gets: re-run order-sensitive generations 50x.
!python benchmark/determinism_audit.py --scores results/B/scores.json --out results/B/determinism_audit.json | tail -n 25

In [ ]:
# 7. Summary: metrics + how closely the GPU-index retrieval matches the committed CPU retrieval eval.
!python scripts/compare_retrieval.py --arm B
!python -c "import json; m=json.load(open('results/B/scores.json'))['metrics']; print('pass@1', m['pass'], '/', m['total'], '=', m['pass_at_1']); print(m['status_counts']); print('by difficulty', {k: v['pass_at_1'] for k, v in m['pass_at_1_by_difficulty'].items()}); print('by category', {k: v['pass_at_1'] for k, v in m['pass_at_1_by_category'].items()})"

In [ ]:
# 8. Zip results for download.
!zip -q -r results_B.zip results/B results/rag
from google.colab import files
files.download('results_B.zip')